# Tesla vs BYD — Financial Performance Analysis

Συγκριτική ανάλυση οικονομικών δεδομένων **Tesla** και **BYD** με Python,  
και οπτικοποίηση αποτελεσμάτων μέσω **Power BI dashboard**.

---

## Μετρικές που αναλύονται

| Μετρική | Περιγραφή |
|---|---|
| **Total Revenue** | Συνολικά έσοδα ανά έτος (B USD) |
| **Revenue Growth %** | Ποσοστιαία μεταβολή εσόδων έτος προς έτος |
| **Gross Profit Margin %** | Μικτό περιθώριο κέρδους |
| **Revenue Difference** | Απόλυτη διαφορά εσόδων Tesla − BYD (B USD) |

## Πηγές δεδομένων
- [Tesla Investor Relations](https://ir.tesla.com/)
- [BYD Annual Reports](https://www.bydglobal.com/en/InvestorRelations.html)
- [Macrotrends](https://www.macrotrends.net/)

## Πώς συνδέονται Python & Power BI

```
Python script
    │
    │  καθαρισμός, μετατροπή νομίσματος, υπολογισμός μετρικών
    ▼
output/financial_trend.xlsx
    │
    │  Get Data → Excel
    ▼
Power BI Dashboard
```

---
## 1. Εισαγωγή βιβλιοθηκών

In [ ]:
import os
import pandas as pd

print('✅ Βιβλιοθήκες φορτώθηκαν επιτυχώς')
print(f'   pandas : {pd.__version__}')

---
## 2. Configuration

Τα δεδομένα BYD είναι σε **CNY (Κινεζικά Γουάν)**.  
Τα μετατρέπουμε σε USD με σταθερό FX rate για σύγκριση με Tesla.

In [ ]:
# FX Rate: CNY → USD
CNY_TO_USD = 1 / 7.2  # προσάρμοσε αν χρειαστεί

# Paths
BASE_DIR   = os.getcwd()
DATA_DIR   = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

TESLA_CSV   = os.path.join(DATA_DIR, 'Tesla_Financials.csv')
BYD_CSV     = os.path.join(DATA_DIR, 'Byd_Financials.csv')
OUTPUT_XLSX = os.path.join(OUTPUT_DIR, 'financial_trend.xlsx')

REQUIRED_COLS = ['Total Revenue', 'Gross Profit']

print(f'📁 Data dir   : {DATA_DIR}')
print(f'📁 Output dir : {OUTPUT_DIR}')
print(f'💱 FX Rate    : 1 CNY = {CNY_TO_USD:.4f} USD')

---
## 3. Φόρτωση & Καθαρισμός Δεδομένων

Η συνάρτηση κάνει τα εξής βήματα:
1. Διαβάζει το CSV και κάνει **transpose** (έτη → index)
2. Αφαιρεί γραμμές **TTM** (Trailing Twelve Months)
3. Εξάγει μόνο το **4ψήφιο έτος** από το index
4. Μετατρέπει το νόμισμα **CNY → USD** (μόνο για BYD)
5. Μετατρέπει από **χιλιάδες → δισεκατομμύρια USD**

In [ ]:
def load_financials(filepath: str, company: str, currency: str) -> pd.DataFrame:
    """
    Φορτώνει CSV οικονομικών δεδομένων.
    - currency: 'USD' για Tesla, 'CNY' για BYD
    - Επιστρέφει DataFrame σε Billions USD με έτος ως index.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'{company} file not found: {filepath}')

    df = pd.read_csv(filepath, sep=';', engine='python')

    # Reshape: Breakdown → columns, Years → index
    df = df.set_index('Breakdown').T
    df.index.name = 'Year'

    # Αφαίρεση TTM γραμμών
    df = df[~df.index.astype(str).str.contains('TTM', na=False)]

    # Εξαγωγή 4ψήφιου έτους
    df.index = df.index.astype(str).str.extract(r'(\d{4})')[0]

    # Καθαρισμός αριθμητικών στηλών
    for col in REQUIRED_COLS:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(',', '', regex=False)
            .astype(float)
        )

    # Μετατροπή CNY → USD (μόνο για BYD)
    if currency == 'CNY':
        df[REQUIRED_COLS] = df[REQUIRED_COLS] * CNY_TO_USD

    # Χιλιάδες → πραγματικές τιμές → Billions USD
    df[REQUIRED_COLS] = df[REQUIRED_COLS] * 1000 / 1e9

    # Ταξινόμηση κατά έτος (κρίσιμο για σωστό Growth %)
    df = df.sort_index()

    return df


tesla = load_financials(TESLA_CSV, 'Tesla', currency='USD')
byd   = load_financials(BYD_CSV,   'BYD',   currency='CNY')

print('Tesla (B USD):')
display(tesla[REQUIRED_COLS])

print('\nBYD (B USD — μετά από CNY→USD):')
display(byd[REQUIRED_COLS])

---
## 4. Υπολογισμός Μετρικών

- **Revenue Growth %** = ποσοστιαία μεταβολή εσόδων έτος προς έτος
- **Gross Profit Margin %** = Gross Profit / Total Revenue × 100

In [ ]:
def compute_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Υπολογίζει Revenue Growth % και Gross Profit Margin %."""
    df = df.copy()
    df['Revenue Growth %'] = df['Total Revenue'].pct_change() * 100
    df['Profit Margin %']  = (df['Gross Profit'] / df['Total Revenue']) * 100
    return df


tesla = compute_metrics(tesla)
byd   = compute_metrics(byd)

print('Tesla — μετρικές:')
display(tesla[['Total Revenue', 'Revenue Growth %', 'Profit Margin %']])

print('\nBYD — μετρικές:')
display(byd[['Total Revenue', 'Revenue Growth %', 'Profit Margin %']])

---
## 5. Συνδυασμός Δεδομένων (Combined DataFrame)

In [ ]:
def build_combined(tesla: pd.DataFrame, byd: pd.DataFrame) -> pd.DataFrame:
    """Συνδυάζει τα δεδομένα των δύο εταιρειών σε ένα DataFrame."""
    combined = pd.DataFrame({
        'Tesla Revenue (B USD)':    tesla['Total Revenue'],
        'Tesla Growth %':           tesla['Revenue Growth %'],
        'Tesla Margin %':           tesla['Profit Margin %'],

        'BYD Revenue (B USD)':      byd['Total Revenue'],
        'BYD Growth %':             byd['Revenue Growth %'],
        'BYD Margin %':             byd['Profit Margin %'],
    })

    combined['Revenue Difference (B USD)'] = (
        combined['Tesla Revenue (B USD)'] - combined['BYD Revenue (B USD)']
    )

    return combined


combined = build_combined(tesla, byd)

print('📈 Combined Dataset:')
display(combined)

---
## 6. Export Excel για Power BI

Το αρχείο `financial_trend.xlsx` εισάγεται στο Power BI:  
**Get Data → Excel Workbook**

In [ ]:
def export_to_powerbi(df: pd.DataFrame, path: str) -> pd.DataFrame:
    """Εξάγει το combined DataFrame σε Excel για χρήση στο Power BI."""
    df = df.copy()
    df.index.name = 'Year'
    df = df.reset_index()
    df.to_excel(path, index=False)
    print(f'✅ Excel dataset exported → {path}')
    return df


df_exported = export_to_powerbi(combined, OUTPUT_XLSX)

print('\nΠροεπισκόπηση exported data:')
display(df_exported)

---
## 7. Συμπεράσματα

*(Συμπλήρωσε εδώ τα συμπεράσματά σου αφού δεις τα γραφήματα στο Power BI)*

- 📈 **Revenue Growth:** ...
- 💰 **Profit Margin:** ...
- ⚡ **Ανταγωνιστικό πλεονέκτημα:** ...

---

## 8. Power BI Dashboard

Το `output/financial_trend.xlsx` που παράχθηκε εισάγεται στο Power BI:

1. Άνοιξε το `dashboard/Tesla_vs_BYD.pbix`
2. **Home → Transform Data → Data Source Settings**
3. Άλλαξε τη διαδρομή ώστε να δείχνει στο τοπικό σου `output/financial_trend.xlsx`
4. **Refresh**